# Interview Outcome Classification with BERT (5 Labels)

Fine-tuning **ruBERT-base** to automatically classify the outcome of a recruitment phone call into one of five categories.

## Label Schema

| Label (model) | Original index | Meaning |
|:---:|:---:|---|
| 0 | 1 | Candidate declined |
| 1 | 2 | Recruiter declined the candidate |
| 2 | 3 | Candidate agreed |
| 3 | 4 | Next steps were discussed |
| 4 | 5 | Candidate actually started the job |

> **Note:** Original index 0 (*call failed for technical reasons*) is excluded from training.

## 1. Installation

In [ ]:
!pip install -q transformers datasets

## 2. Imports

In [ ]:
import os
import gc
import glob

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm, trange
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from torch.utils.data import DataLoader
from torch.optim import AdamW

## 3. Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────
DATA_DIR   = "/content/drive/MyDrive/Colab Notebooks/Eurasia Bert for interviews evaluation/dataset 5_cat"
MODEL_DIR  = os.path.join(DATA_DIR, "interview_classifier")

# ── Model ────────────────────────────────────────────────────────────────
BASE_MODEL = "ai-forever/ruBert-base"  # https://huggingface.co/ai-forever/ruBert-base

# ── Training hyperparameters ─────────────────────────────────────────────
NUM_LABELS  = 5
BATCH_SIZE  = 4
MAX_LENGTH  = 512
LEARNING_RATE = 1e-6   # keep small for tiny batches
NUM_EPOCHS  = 3
TEST_SIZE   = 0.2
RANDOM_SEED = 42

# ── Labels ───────────────────────────────────────────────────────────────
LABEL_NAMES = [
    "Candidate declined",
    "Recruiter declined candidate",
    "Candidate agreed",
    "Next steps discussed",
    "Candidate started job",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 4. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Load & Merge Data

In [ ]:
os.chdir(DATA_DIR)

csv_files = glob.glob("*.csv")
print(f"Found {len(csv_files)} CSV file(s): {csv_files}")

df_raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print(f"Total rows loaded: {len(df_raw):,}")

## 6. Data Preprocessing

In [ ]:
df = df_raw.copy()
df = df.rename(columns={"result_column": "label"})

# ── Missing values ───────────────────────────────────────────────────────
missing = df["label"].isna().sum()
print(f"Missing labels: {missing}")
df["label"] = df["label"].fillna(0)

# ── Cast to int ──────────────────────────────────────────────────────────
df["label"] = df["label"].astype(int)

print("\nLabel distribution (all data):")
print(df["label"].value_counts().sort_index())

In [ ]:
# Preview a few samples per class
pd.options.display.max_colwidth = 300
print(f"Dataset shape: {df.shape}")
df.groupby("label").sample(2, random_state=RANDOM_SEED)

## 7. Build Hugging Face Dataset

In [ ]:
# Exclude label 0 (call failed for technical reasons — not a classification outcome)
df_train = df[df["label"] > 0].copy()

# Shift labels to 0-based: original 1-5 → model 0-4
df_train["label"] = df_train["label"] - 1

print("Training label distribution:")
print(df_train["label"].value_counts().sort_index())

hf_dataset = Dataset.from_dict(
    {"text": df_train["text"].tolist(), "label": df_train["label"].tolist()}
).train_test_split(test_size=TEST_SIZE, seed=RANDOM_SEED)

print("\nDataset splits:")
print(hf_dataset)

## 8. Tokenisation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = hf_dataset.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized)

## 9. DataLoaders

In [ ]:
collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    tokenized["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=collator
)
val_loader = DataLoader(
    tokenized["test"], shuffle=False, batch_size=BATCH_SIZE, collate_fn=collator
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 10. Model

We use [`BertForSequenceClassification`](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertForSequenceClassification) from `transformers`.  
It adds a dropout + linear head on top of the `[CLS]` token representation and computes cross-entropy loss when labels are provided.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=NUM_LABELS
)
model.to(DEVICE)
print(model.config)

## 11. Optimizer

In [ ]:
# AdamW is generally preferred over plain Adam for transformers
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Free unused GPU memory before training
gc.collect()
torch.cuda.empty_cache()

## 12. Training Loop

In [ ]:
def evaluate(loader):
    """Run evaluation; return (mean_loss, predictions, targets)."""
    model.eval()
    losses, preds, targets = [], [], []
    for batch in tqdm(loader, desc="Eval", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.no_grad():
            out = model(**batch)
        losses.append(out.loss.item())
        preds.extend(out.logits.argmax(1).tolist())
        targets.extend(batch["labels"].tolist())
    return np.mean(losses), preds, targets


train_losses = []

for epoch in trange(NUM_EPOCHS, desc="Epochs"):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=True)

    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        out.loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        train_losses.append(out.loss.item())
        pbar.set_description(f"Epoch {epoch+1} | loss: {np.mean(train_losses[-100:]):.3f}")

    val_loss, val_preds, val_targets = evaluate(val_loader)
    acc = np.mean(np.array(val_targets) == np.array(val_preds))
    print(
        f"[Epoch {epoch+1}] "
        f"train_loss={np.mean(train_losses[-100:]):.3f}  "
        f"val_loss={val_loss:.3f}  "
        f"val_acc={acc:.3f}"
    )

## 13. Evaluation

In [ ]:
# Loss curve
plt.figure(figsize=(10, 3))
plt.plot(train_losses, alpha=0.6, label="Train loss (per batch)")
plt.xlabel("Batch")
plt.ylabel("Cross-entropy loss")
plt.title("Training Loss Curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
_, val_preds, val_targets = evaluate(val_loader)

print("Classification Report:")
print(classification_report(val_targets, val_preds, target_names=LABEL_NAMES))

print("Confusion Matrix:")
print(confusion_matrix(val_targets, val_preds))

## 14. Save Model

In [ ]:
# Ensure tensors are contiguous before saving
for param in model.parameters():
    param.data = param.data.contiguous()

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"Model saved to: {MODEL_DIR}")
!ls "{MODEL_DIR}" -alsh

## 15. Load Model & Run Inference

In [ ]:
# Load from disk (useful when resuming from a saved checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

In [ ]:
def classify(text: str) -> dict:
    """
    Classify a single text string.

    Returns a dict mapping each label name to its predicted probability.
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    return {name: float(f"{p:.4f}") for name, p in zip(LABEL_NAMES, probs)}

In [ ]:
# Example predictions
test_phrases = [
    "We don't have any positions matching your profile",  # recruiter declines
    "We'll be in touch with you",                         # ambiguous / next steps
    "Yes, I'm ready to start on Monday",                  # candidate agreed
]

for phrase in test_phrases:
    result = classify(phrase)
    top = max(result, key=result.get)
    print(f"\nText : {phrase}")
    print(f"Top  : {top} ({result[top]:.1%})")
    for label, prob in result.items():
        print(f"  {label:<30} {prob:.1%}")